# Vocabulary sweep for topic modelling

### Vocabulary sweep pipeline overview

Input: `checkpoints/df_with_noun_tokens.pkl`, `data/fashion_ontology.csv`

Output: `checkpoints/chosen_vocab.pkl`, `outputs_eda/`

- **1. Setup:** load noun-token checkpoint and ontology lookup sets
- **2. Diagnostics:** build n-gram tables, tag fashion vs. non-fashion terms, inspect corpus-level term frequencies
- **3. Vocabulary sweep:** run all combinations of `min_df` / `max_df` / TF-IDF percentile at a fixed k; compare vocab size, fashion-term retention, and coherence
- **4. Topic count sweep:** fix chosen vocabulary parameters, sweep `TOPIC_RANGE` to find the best k by C_v coherence
- **5. Outputs:** save sweep results and chosen vocabulary checkpoint for downstream modelling

### Key variables
- `ONTOLOGY_UNIGRAMS / BIGRAMS / TRIGRAMS`: term sets used to tag fashion vs. non-fashion tokens
- `vocab_sweep_df`: results table from Stage A (one row per parameter combination)
- `chosen_vocab`: dict containing the locked vocabulary, count matrix, and token lists
- `topic_sweep_df`: coherence and perplexity at each k

# 1. Setup

In [1]:
from __future__ import annotations

from pathlib import Path
from itertools import product as iterproduct

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pickle

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel


In [2]:
# Cell 1: Global variables and parameters

# Paths
CHECKPOINT_DIR = Path("checkpoints")
OUT_DIR        = Path("outputs_eda")
OUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Sweep parameters

## Vocabulary boundary
MIN_DF_VALUES             = [2, 5, 10, 15, 20]
MAX_DF_PROP               = 0.95           # fixed: no term exceeds 19% doc-frequency in this corpus
TFIDF_PERCENTILES_TO_DROP = [0, 30, 50]   # applied post-min_df so it cuts independently of min_df
FIXED_K_FOR_VOCAB_SWEEP   = 20

TFIDF_PERCENTILES_TO_DROP = [0, 25, 50, 75]

## Vocabulary sweep max iterations
VOCAB_SWEEP_MAX_ITER  = 30

# Topic count
TOPIC_RANGE           = list(range(10, 56, 5))   # 10, 15, …, 55
TOPIC_SWEEP_MAX_ITER  = 50

RANDOM_STATE = 42
N_TOP_WORDS  = 15

In [3]:
# Cell 2: Load noun-token checkpoint
checkpoint_path = CHECKPOINT_DIR / "df_with_noun_tokens.pkl"

if not checkpoint_path.exists():
    raise FileNotFoundError(
        f"{checkpoint_path} not found — run notebook 2 first."
    )

df = pd.read_pickle(checkpoint_path)
print(f"Loaded checkpoint: {df.shape}")

# Sanity check
assert "noun_tokens" in df.columns, "Missing column: noun_tokens"
assert "noun_text"   in df.columns, "Missing column: noun_text"

_sample = df["noun_tokens"].dropna().head(200)
_joined = _sample.apply(lambda toks: " ".join(t.lower() for t in toks))
_stored = df.loc[_sample.index, "noun_text"].str.lower().str.strip()
assert (_joined == _stored).all(), (
    "noun_text and noun_tokens out of sync"
)
print("noun_text / noun_tokens consistency OK")


Loaded checkpoint: (6629, 13)
noun_text / noun_tokens consistency OK


# 2. Diagnostics

In [4]:
# Cell 3: Load ontology and build term sets
ONTOLOGY_CSV = Path("data/fashion_ontology.csv")
if not ONTOLOGY_CSV.exists():
    raise FileNotFoundError(f"{ONTOLOGY_CSV} not found — run notebook 2 first.")

ontology_df = pd.read_csv(ONTOLOGY_CSV)
ontology_df["term"]     = ontology_df["term"].astype(str).str.strip().str.lower()
ontology_df["category"] = ontology_df["category"].astype(str).str.strip().str.lower()
if "n_words" not in ontology_df.columns:
    ontology_df["n_words"] = ontology_df["term"].str.split().str.len()

ONTOLOGY_UNIGRAMS = set(ontology_df.loc[ontology_df["n_words"] == 1, "term"])
ONTOLOGY_BIGRAMS  = set(ontology_df.loc[ontology_df["n_words"] == 2, "term"])
ONTOLOGY_TRIGRAMS = set(ontology_df.loc[ontology_df["n_words"] == 3, "term"])
ONTOLOGY_ALL      = ONTOLOGY_UNIGRAMS | ONTOLOGY_BIGRAMS | ONTOLOGY_TRIGRAMS
TERM_TO_CATEGORY  = dict(zip(ontology_df["term"], ontology_df["category"]))

print(
    f"Ontology: {len(ONTOLOGY_UNIGRAMS)} unigrams, "
    f"{len(ONTOLOGY_BIGRAMS)} bigrams, "
    f"{len(ONTOLOGY_TRIGRAMS)} trigrams"
)


Ontology: 719 unigrams, 491 bigrams, 106 trigrams


In [5]:
# Cell 4: Build n-gram occurrence table (diagnostic only; bigrams/trigrams are NOT fed to LDA)
# Note: make_ngrams is defined once here and used only in this section.

def make_ngrams(tokens: list[str], n: int) -> list[str]:
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


ngram_rows = []
for _, row in df[["doc_id", "noun_tokens"]].iterrows():
    toks = row["noun_tokens"]
    for term in toks:
        ngram_rows.append({"doc_id": row["doc_id"], "term": term, "n": 1})
    for term in make_ngrams(toks, 2):
        ngram_rows.append({"doc_id": row["doc_id"], "term": term, "n": 2})
    for term in make_ngrams(toks, 3):
        ngram_rows.append({"doc_id": row["doc_id"], "term": term, "n": 3})

ngram_df = pd.DataFrame(ngram_rows)
print(f"N-gram table: {len(ngram_df):,} rows")
ngram_df.head()


N-gram table: 1,157,019 rows


,doc_id,term,n
0,0,sophomore,1
1,0,coed,1
2,0,sailor,1
3,0,simple,1
4,0,cotton,1


In [6]:
# Cell 5: Tag each n-gram as fashion-related via ontology set membership
_ontology_by_n = {1: ONTOLOGY_UNIGRAMS, 2: ONTOLOGY_BIGRAMS, 3: ONTOLOGY_TRIGRAMS}

ngram_df["term_lower"]      = ngram_df["term"].str.lower()
ngram_df["is_fashion"]      = ngram_df.apply(
    lambda r: r["term_lower"] in _ontology_by_n.get(r["n"], set()), axis=1
)
ngram_df["fashion_category"] = ngram_df["term_lower"].map(TERM_TO_CATEGORY).fillna("non_fashion")

ngram_df.head()


,doc_id,term,n,term_lower,is_fashion,fashion_category
0,0,sophomore,1,sophomore,True,brand
1,0,coed,1,coed,False,non_fashion
2,0,sailor,1,sailor,True,garment
3,0,simple,1,simple,False,non_fashion
4,0,cotton,1,cotton,True,material


In [7]:
# Cell 6: Corpus-level term frequency tables for manual inspection
term_counts = (
    ngram_df.groupby(["n", "term", "is_fashion", "fashion_category"])
    .size()
    .reset_index(name="corpus_freq")
    .sort_values(["n", "corpus_freq"], ascending=[True, False])
)

top_kept_fashion_terms     = term_counts[term_counts["is_fashion"]].copy()
top_kept_non_fashion_terms = term_counts[~term_counts["is_fashion"]].copy()
matched_ontology_bigrams   = term_counts[(term_counts["n"] == 2) & term_counts["is_fashion"]].copy()
matched_ontology_trigrams  = term_counts[(term_counts["n"] == 3) & term_counts["is_fashion"]].copy()
ambiguous_high_freq_terms  = term_counts[(~term_counts["is_fashion"]) & (term_counts["corpus_freq"] >= 10)].copy()

print(f"Unique unigrams in corpus: {(term_counts.n == 1).sum():,}")
print(f"Fashion unigrams matched:  {((term_counts.n == 1) & term_counts['is_fashion']).sum():,}")
top_kept_non_fashion_terms.head(20)


Unique unigrams in corpus: 18,869
Fashion unigrams matched:  453


,n,term,is_fashion,fashion_category,corpus_freq
12218,1,point,False,non_fashion,1155
1513,1,bit,False,non_fashion,1134
1461,1,big,False,non_fashion,1097
14713,1,signature,False,non_fashion,1070
7313,1,hand,False,non_fashion,1028
1659,1,body,False,non_fashion,1015
9399,1,lot,False,non_fashion,979
1859,1,brand,False,non_fashion,962
8905,1,label,False,non_fashion,951
15805,1,strong,False,non_fashion,941


In [8]:
# Cell 7: Save baseline EDA tables
term_counts.to_csv(OUT_DIR / "baseline_term_counts.csv", index=False)
top_kept_fashion_terms.to_csv(OUT_DIR / "baseline_top_kept_fashion_terms.csv", index=False)
top_kept_non_fashion_terms.to_csv(OUT_DIR / "baseline_top_kept_non_fashion_terms.csv", index=False)
matched_ontology_bigrams.to_csv(OUT_DIR / "baseline_matched_ontology_bigrams.csv", index=False)
matched_ontology_trigrams.to_csv(OUT_DIR / "baseline_matched_ontology_trigrams.csv", index=False)
ambiguous_high_freq_terms.to_csv(OUT_DIR / "baseline_ambiguous_high_freq_terms.csv", index=False)

print(f"Saved baseline tables to: {OUT_DIR.resolve()}")


Saved baseline tables to: /Users/zoeoggel/Data Science/Thesis/Thesis-Git/outputs_eda


In [9]:
# Cell 8: Baseline vectorisers (no thresholding), for TF-IDF diagnostic only
# noun_text is used here as the input string; noun_tokens drives everything else.
docs = df["noun_text"].fillna("").tolist()

base_count_vec = CountVectorizer(lowercase=False, token_pattern=r"(?u)\b\w+\b")
base_tfidf_vec = TfidfVectorizer(lowercase=False, token_pattern=r"(?u)\b\w+\b")

X_count_base = base_count_vec.fit_transform(docs)
X_tfidf_base = base_tfidf_vec.fit_transform(docs)

count_terms = np.array(base_count_vec.get_feature_names_out())

print(f"Baseline count matrix: {X_count_base.shape}")
print(f"Baseline TF-IDF matrix: {X_tfidf_base.shape}")


Baseline count matrix: (6629, 18869)
Baseline TF-IDF matrix: (6629, 18869)


In [10]:
# Cell 9: Term-level statistics from baseline (used to inform TF-IDF percentile cut)
term_doc_freq    = np.asarray((X_count_base > 0).sum(axis=0)).ravel()
term_corpus_freq = np.asarray(X_count_base.sum(axis=0)).ravel()
term_mean_tfidf  = np.asarray(X_tfidf_base.mean(axis=0)).ravel()
term_max_tfidf   = np.asarray(X_tfidf_base.max(axis=0).toarray()).ravel()

term_stats = pd.DataFrame({
    "term":          count_terms,
    "doc_freq":      term_doc_freq,
    "doc_freq_prop": term_doc_freq / len(df),
    "corpus_freq":   term_corpus_freq,
    "mean_tfidf":    term_mean_tfidf,
    "max_tfidf":     term_max_tfidf,
})
term_stats["is_fashion"]       = term_stats["term"].str.lower().isin(ONTOLOGY_UNIGRAMS)
term_stats["fashion_category"] = term_stats["term"].str.lower().map(TERM_TO_CATEGORY).fillna("non_fashion")

term_stats.sort_values("doc_freq_prop", ascending=False).head(30)


,term,doc_freq,doc_freq_prop,corpus_freq,mean_tfidf,max_tfidf,is_fashion,fashion_category
7583,high,1253,0.189018,1438,0.013601,0.237074,True,garment
9357,long,1226,0.184945,1441,0.014140,0.257430,True,style
14612,shoulder,1105,0.166692,1258,0.012931,0.229741,True,garment
13220,red,1066,0.160809,1302,0.013284,0.324473,True,color
12218,point,1041,0.157037,1155,0.011547,0.212765,False,non_fashion
5461,evening,1041,0.157037,1168,0.012304,0.264974,True,occasion
14713,signature,1033,0.155830,1070,0.011167,0.201396,False,non_fashion
14937,sleeve,1004,0.151456,1107,0.011777,0.264307,True,garment
1513,bit,982,0.148137,1134,0.011929,0.252564,False,non_fashion
8913,lace,975,0.147081,1257,0.013084,0.341344,True,material


In [11]:
# Cell 10: Plot TF-IDF distribution to inform percentile cut choice
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(term_stats["mean_tfidf"], bins=100, edgecolor="none")
axes[0].set_xlabel("mean TF-IDF")
axes[0].set_ylabel("# terms")
axes[0].set_title("Mean TF-IDF — all terms")

cutoff_20 = term_stats["mean_tfidf"].quantile(0.20)
low = term_stats[term_stats["mean_tfidf"] <= cutoff_20]
axes[1].hist(low["mean_tfidf"], bins=60, edgecolor="none", color="steelblue")
axes[1].set_xlabel("mean TF-IDF")
axes[1].set_title("Bottom 20% — inspect to choose percentile cut")

plt.tight_layout()
plt.savefig(OUT_DIR / "tfidf_distribution.png", dpi=120)
plt.show()

print(
    "Inspect the right panel. If the lowest-TF-IDF terms are uninformative noise, "
    "increase TFIDF_PERCENTILES_TO_DROP in the parameters cell above."
)


Inspect the right panel. If the lowest-TF-IDF terms are uninformative noise, increase TFIDF_PERCENTILES_TO_DROP in the parameters cell above.


/var/folders/_p/f1mg9b290kx6bf1ytjkd60qm0000gn/T/ipykernel_77502/3889872546.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# 3. Vocabulary sweep

In [24]:
# Cell 11: Build vocabulary function with three filters

def build_vocabulary(
    min_df: int,
    max_df_prop: float,
    tfidf_pct_to_drop: float,
) -> dict:
    """
    Build a vocabulary and count matrix from noun_tokens using three filters (max_df_prop, min_df, and TF-IDF percentile cut)
    
    Returns a dict suitable for fit_lda_single and the checkpoint.
    """
    
    # Filter 1: max_df, pre-exclude corpus-wide stop terms, fixed threshold
    # Fixed this after some consideration but choosing to keep it in this function
    excluded_by_max_df = set(
        term_stats.loc[term_stats["doc_freq_prop"] > max_df_prop, "term"].str.lower()
    )

    filtered_docs = [
        " ".join(t.lower() for t in toks if t.lower() not in excluded_by_max_df)
        for toks in df["noun_tokens"]
    ]

    # Filter 2: min_df via CountVectorizer
    count_vec     = CountVectorizer(lowercase=False, token_pattern=r"(?u)\b\w+\b", min_df=min_df)
    X_count       = count_vec.fit_transform(filtered_docs)
    feature_names = np.array(count_vec.get_feature_names_out())

    # Filter 3: TF-IDF applied to the post-min_df vocabulary
    if tfidf_pct_to_drop > 0:
        tfidf_vec = TfidfVectorizer(
            lowercase=False,
            token_pattern=r"(?u)\b\w+\b",
            vocabulary={t: i for i, t in enumerate(feature_names)},
        )
        X_tfidf_post  = tfidf_vec.fit_transform(filtered_docs)
        mean_tfidf_post = np.asarray(X_tfidf_post.mean(axis=0)).ravel()
        tfidf_cutoff  = float(np.percentile(mean_tfidf_post, tfidf_pct_to_drop))
        keep_mask     = mean_tfidf_post > tfidf_cutoff
        feature_names = feature_names[keep_mask]
        X_count       = X_count[:, keep_mask]
    else:
        tfidf_cutoff = 0.0

    kept_terms = set(feature_names)

    # Token lists for coherence
    token_lists_all = [
        [t.lower() for t in toks if t.lower() in kept_terms]
        for toks in df["noun_tokens"]
    ]
    n_empty = sum(1 for tl in token_lists_all if not tl)
    token_lists = [tl for tl in token_lists_all if tl]

    # Fashion share
    kept_fashion_share = float(np.mean([t in ONTOLOGY_UNIGRAMS for t in kept_terms]))

    # Candidate pool for reporting n_excluded
    all_candidate  = set(pd.Series(" ".join(filtered_docs).split()).str.lower().unique())
    final_excluded = excluded_by_max_df | (all_candidate - kept_terms)

    return {
        "min_df":             min_df,
        "max_df_prop":        max_df_prop,
        "tfidf_pct_to_drop":  tfidf_pct_to_drop,
        "tfidf_cutoff":       tfidf_cutoff,
        "X_count":            X_count,
        "feature_names":      feature_names,
        "kept_terms":         kept_terms,
        "final_excluded":     final_excluded,
        "filtered_docs":      filtered_docs,
        "token_lists":        token_lists,
        "n_empty_docs":       n_empty,
        "vocab_size":         len(kept_terms),
        "n_excluded":         len(final_excluded),
        "kept_fashion_share": kept_fashion_share,
    }

In [13]:
# Cell 12: Helper functions for LDA fitting

def get_top_words_per_topic(
    lda_model: LatentDirichletAllocation,
    feature_names: np.ndarray,
    n_top_words: int = N_TOP_WORDS,
) -> pd.DataFrame:
    """Extract top words for each topic from a fitted LDA model."""
    rows = []
    for idx, topic in enumerate(lda_model.components_):
        top_idx = topic.argsort()[:-n_top_words - 1:-1]
        words   = [feature_names[i] for i in top_idx]
        rows.append({"topic_id": idx, "top_words": ", ".join(words), "top_words_list": words})
    return pd.DataFrame(rows)


def compute_cv_coherence(
    topic_word_lists: list[list[str]], 
    token_lists: list[list[str]]
) -> float:
    """Compute C_v coherence for a set of topics and tokenized documents."""
    dictionary = Dictionary(token_lists)
    corpus     = [dictionary.doc2bow(t) for t in token_lists]
    return CoherenceModel(
        topics=topic_word_lists, texts=token_lists,
        corpus=corpus, dictionary=dictionary, coherence="c_v",
    ).get_coherence()


def fit_lda_single(
    vocab_dict: dict,
    k: int,
    max_iter: int = TOPIC_SWEEP_MAX_ITER,
    random_state: int = RANDOM_STATE,
) -> tuple:
    """Fit one LDA model and return (lda, topics_df, coherence_cv, perplexity)."""
    lda = LatentDirichletAllocation(
        n_components=k,
        random_state=random_state,
        learning_method="batch",
        max_iter=max_iter,
        evaluate_every=-1,
    )
    lda.fit(vocab_dict["X_count"])
    topics_df    = get_top_words_per_topic(lda, vocab_dict["feature_names"])
    coherence_cv = compute_cv_coherence(
        topics_df["top_words_list"].tolist(), vocab_dict["token_lists"]
    )
    perplexity   = lda.perplexity(vocab_dict["X_count"])
    return lda, topics_df, coherence_cv, perplexity


In [ ]:
# Cell 13: Vocabulary sweep, incremental CSV write

VOCAB_SWEEP_CSV = OUT_DIR / "vocab_sweep_results.csv"

# Set to True to discard existing results
RESTART = False


if VOCAB_SWEEP_CSV.exists() and not RESTART:
    _existing = pd.read_csv(VOCAB_SWEEP_CSV)
    _done = set(zip(_existing["min_df"], _existing["tfidf_pct_to_drop"]))
    print(f"Resuming: {len(_done)} combinations already done.")
else:
    _existing = pd.DataFrame()
    _done = set()
    pd.DataFrame(columns=[
        "min_df", "tfidf_pct_to_drop", "tfidf_cutoff",
        "vocab_size", "n_excluded", "n_empty_docs", "kept_fashion_share",
        f"coherence_cv_k{FIXED_K_FOR_VOCAB_SWEEP}",
        f"perplexity_k{FIXED_K_FOR_VOCAB_SWEEP}",
    ]).to_csv(VOCAB_SWEEP_CSV, index=False)

combos = list(iterproduct(MIN_DF_VALUES, TFIDF_PERCENTILES_TO_DROP))
total  = len(combos)

for i, (min_df, pct) in enumerate(combos, 1):
    if (min_df, pct) in _done:
        continue
    print(f"[{i}/{total}] min_df={min_df}  tfidf_drop={pct}%", end=" ... ")
    v = build_vocabulary(min_df, MAX_DF_PROP, pct)
    _, _, coherence_cv, perplexity = fit_lda_single(
        v, k=FIXED_K_FOR_VOCAB_SWEEP, max_iter=VOCAB_SWEEP_MAX_ITER
    )
    row = {
        "min_df":             min_df,
        "tfidf_pct_to_drop":  pct,
        "tfidf_cutoff":       v["tfidf_cutoff"],
        "vocab_size":         v["vocab_size"],
        "n_excluded":         v["n_excluded"],
        "n_empty_docs":       v["n_empty_docs"],
        "kept_fashion_share": v["kept_fashion_share"],
        f"coherence_cv_k{FIXED_K_FOR_VOCAB_SWEEP}": coherence_cv,
        f"perplexity_k{FIXED_K_FOR_VOCAB_SWEEP}":   perplexity,
    }
    pd.DataFrame([row]).to_csv(VOCAB_SWEEP_CSV, mode="a", header=False, index=False)
    print(f"vocab={v['vocab_size']}  empty_docs={v['n_empty_docs']}  fashion={v['kept_fashion_share']:.3f}  coh={coherence_cv:.4f}")

vocab_sweep_df = (
    pd.read_csv(VOCAB_SWEEP_CSV)
    .sort_values(f"coherence_cv_k{FIXED_K_FOR_VOCAB_SWEEP}", ascending=False)
    .reset_index(drop=True)
)
vocab_sweep_df

Resuming: 20 combinations already done.


,min_df,tfidf_pct_to_drop,tfidf_cutoff,vocab_size,n_excluded,n_empty_docs,kept_fashion_share,coherence_cv_k20,perplexity_k20
0,20,75,0.002401,767,18102,4,0.219035,0.434883,726.416001
1,10,75,0.001533,1175,17694,4,0.180426,0.433999,1016.447396
2,15,75,0.002019,917,17952,4,0.200654,0.433225,839.685443
3,15,50,0.001038,1833,17036,4,0.133661,0.430243,1421.523840
4,20,25,0.000822,2299,16570,4,0.117877,0.425456,1685.853134
5,10,50,0.000746,2349,16520,4,0.117923,0.423566,1703.409800
6,20,50,0.001281,1533,17336,4,0.151990,0.421563,1246.402728
7,10,25,0.000429,3524,15345,4,0.093076,0.421246,2230.450322
8,5,75,0.001006,1747,17122,4,0.139096,0.419610,1374.765687
9,20,0,0.000000,3066,15803,4,0.100457,0.414693,2028.915294


### Choose vocabulary parameters

In [15]:
# Cell 14: Choose vocabulary parameters based on sweep results

_best = vocab_sweep_df.iloc[0]

# CHOSEN_MIN_DF         = int(_best["min_df"])
# CHOSEN_MAX_DF_PROP    = MAX_DF_PROP          # fixed globally — not swept
# CHOSEN_TFIDF_PCT_DROP = float(_best["tfidf_pct_to_drop"])

# Override best coherence score for a higher vocabulary size and higher absolute fashion share
# These are more important for downstream tasks and the difference in coherence is likely insignificant
CHOSEN_MIN_DF         = 20
CHOSEN_MAX_DF_PROP    = MAX_DF_PROP
CHOSEN_TFIDF_PCT_DROP = 50.0



print("Chosen vocabulary parameters:")
print(f"  min_df             = {CHOSEN_MIN_DF}")
print(f"  max_df_prop        = {CHOSEN_MAX_DF_PROP}  (fixed)")
print(f"  tfidf_pct_to_drop  = {CHOSEN_TFIDF_PCT_DROP}")
print(f"  vocab_size         = {int(_best['vocab_size'])}")
print(f"  n_empty_docs       = {int(_best['n_empty_docs'])}")
print(f"  kept_fashion_share = {_best['kept_fashion_share']:.3f}")
print(f"  coherence_cv_k{FIXED_K_FOR_VOCAB_SWEEP}   = {_best[f'coherence_cv_k{FIXED_K_FOR_VOCAB_SWEEP}']:.4f}")

Chosen vocabulary parameters:
  min_df             = 20
  max_df_prop        = 0.95  (fixed)
  tfidf_pct_to_drop  = 50.0
  vocab_size         = 767
  n_empty_docs       = 4
  kept_fashion_share = 0.219
  coherence_cv_k20   = 0.4349


# 4. Topic count sweep

In [16]:
# Cell 15: Build chosen vocabulary
chosen_vocab = build_vocabulary(
    min_df=CHOSEN_MIN_DF,
    max_df_prop=CHOSEN_MAX_DF_PROP,
    tfidf_pct_to_drop=CHOSEN_TFIDF_PCT_DROP,
)

print(f"Vocabulary locked: {chosen_vocab['vocab_size']} terms")
print(f"Document-term matrix: {chosen_vocab['X_count'].shape}")
print(f"Non-empty documents for coherence: {len(chosen_vocab['token_lists'])}")
if chosen_vocab["n_empty_docs"] > 0:
    print(
        f"{chosen_vocab['n_empty_docs']} documents empty after filtering, excluded from coherence computation."
    )

Vocabulary locked: 1533 terms
Document-term matrix: (6629, 1533)
Non-empty documents for coherence: 6625
4 documents empty after filtering, excluded from coherence computation.


In [17]:
# Cell 16: Topic count sweep, incremental CSV write
TOPIC_SWEEP_CSV = OUT_DIR / "topic_sweep_results.csv"

if TOPIC_SWEEP_CSV.exists():
    _ts_existing = pd.read_csv(TOPIC_SWEEP_CSV)
    _ts_done = set(_ts_existing["n_topics"])
    print(f"Resuming: {len(_ts_done)} k values already done.")
else:
    _ts_existing = pd.DataFrame()
    _ts_done = set()
    pd.DataFrame(columns=["n_topics", "coherence_cv", "perplexity"]).to_csv(
        TOPIC_SWEEP_CSV, index=False
    )

total = len(TOPIC_RANGE)
for i, k in enumerate(TOPIC_RANGE, 1):
    if k in _ts_done:
        continue
    print(f"[{i}/{total}] k={k}", end=" ... ")
    _, _, coherence_cv, perplexity = fit_lda_single(
        chosen_vocab, k=k, max_iter=TOPIC_SWEEP_MAX_ITER
    )
    pd.DataFrame([{"n_topics": k, "coherence_cv": coherence_cv, "perplexity": perplexity}]).to_csv(
        TOPIC_SWEEP_CSV, mode="a", header=False, index=False
    )
    print(f"coherence={coherence_cv:.4f}  perplexity={perplexity:.1f}")

topic_sweep_df = pd.read_csv(TOPIC_SWEEP_CSV).sort_values("n_topics").reset_index(drop=True)
topic_sweep_df


Resuming: 3 k values already done.
[4/10] k=25 ... coherence=0.4274  perplexity=1266.2
[5/10] k=30 ... coherence=0.4230  perplexity=1289.6
[6/10] k=35 ... coherence=0.4316  perplexity=1325.6
[7/10] k=40 ... coherence=0.4297  perplexity=1355.1
[8/10] k=45 ... coherence=0.4247  perplexity=1387.8
[9/10] k=50 ... coherence=0.4227  perplexity=1415.6
[10/10] k=55 ... coherence=0.4296  perplexity=1452.1


,n_topics,coherence_cv,perplexity
0,10,0.397768,1167.243701
1,15,0.429118,1199.099068
2,20,0.429313,1236.828393
3,25,0.427382,1266.229966
4,30,0.423018,1289.616902
5,35,0.431634,1325.552308
6,40,0.429746,1355.052883
7,45,0.424672,1387.831139
8,50,0.422712,1415.631904
9,55,0.429598,1452.127342


In [18]:
# Cell 17: Plot coherence
fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()

ax1.plot(topic_sweep_df["n_topics"], topic_sweep_df["coherence_cv"], "b-o", label="C_v coherence")
ax2.plot(topic_sweep_df["n_topics"], topic_sweep_df["perplexity"],   "r--s", label="Perplexity", alpha=0.6)

best_k_auto = int(topic_sweep_df.loc[topic_sweep_df["coherence_cv"].idxmax(), "n_topics"])
ax1.axvline(best_k_auto, color="steelblue", linestyle=":", linewidth=1.5, label=f"Peak coherence k={best_k_auto}")

coh_vals = topic_sweep_df["coherence_cv"].values

ax1.set_xlabel("Number of topics (k)")
ax1.set_ylabel("Coherence (C_v)", color="b")
ax2.set_ylabel("Perplexity", color="r")
ax1.legend(loc="upper left")
ax2.legend(loc="upper right")
plt.title(
    f"Topic sweep — min_df={CHOSEN_MIN_DF}, "
    f"max_df={CHOSEN_MAX_DF_PROP}, tfidf_drop={CHOSEN_TFIDF_PCT_DROP}%"
)
plt.tight_layout()
plt.savefig(OUT_DIR / "topic_sweep_plot.png", dpi=120)
plt.show()

print(f"Peak coherence k : {best_k_auto}")

Peak coherence k : 35


/var/folders/_p/f1mg9b290kx6bf1ytjkd60qm0000gn/T/ipykernel_77502/2642418197.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [22]:
# Cell 18: Choose final k based on sweep results and plot

# CHOSEN_K = best_k_auto

# Highest coherence with lowest perplexity
CHOSEN_K = 15
print(f"Final k = {CHOSEN_K}")


Final k = 15


# 5. Save checkpoint

In [23]:
# Cell 19: Save vocabulary checkpoint 

checkpoint_out = CHECKPOINT_DIR / "chosen_vocab.pkl"

with open(checkpoint_out, "wb") as f:
    pickle.dump(
        {
            # Vocabulary
            "chosen_vocab":          chosen_vocab,
            "CHOSEN_MIN_DF":         CHOSEN_MIN_DF,
            "CHOSEN_MAX_DF_PROP":    CHOSEN_MAX_DF_PROP,
            "CHOSEN_TFIDF_PCT_DROP": CHOSEN_TFIDF_PCT_DROP,
            # Topic selection
            "CHOSEN_K":              CHOSEN_K,
            "topic_sweep_df":        topic_sweep_df,
            "vocab_sweep_df":        vocab_sweep_df,
            # Constants needed downstream
            "ONTOLOGY_UNIGRAMS":     ONTOLOGY_UNIGRAMS,
            "ONTOLOGY_BIGRAMS":      ONTOLOGY_BIGRAMS,
            "ONTOLOGY_TRIGRAMS":     ONTOLOGY_TRIGRAMS,
            "TERM_TO_CATEGORY":      TERM_TO_CATEGORY,
            "RANDOM_STATE":          RANDOM_STATE,
        },
        f,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

print(f"Checkpoint saved: {checkpoint_out.resolve()}")
print(f"  vocab_size = {chosen_vocab['vocab_size']}")
print(f"  CHOSEN_K   = {CHOSEN_K}")


Checkpoint saved: /Users/zoeoggel/Data Science/Thesis/Thesis-Git/checkpoints/chosen_vocab.pkl
  vocab_size = 1533
  CHOSEN_K   = 15
